# Medical CPT Training - Continued Pretraining (Full Model)
## Qwen2.5-7B with Memory Optimization

**Trains the FULL model** (not adapters) using:
- bfloat16 precision (not quantized)
- Gradient checkpointing
- Small batch + accumulation
- Memory-efficient optimizer

Expected GPU memory: 26-30GB (fits on 31.84GB)

In [ ]:
# ============ CELL 1: IMPORTS & SETUP ============
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("MEDICAL CPT - FULL MODEL TRAINING")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

if not torch.cuda.is_available():
    print("❌ CUDA not available!")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"GPU Memory: {gpu_mem:.1f}GB")
print("="*80 + "\n")

In [ ]:
# ============ CELL 2: CHECK PACKAGES ============
print("Checking packages...")
try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        DataCollatorForLanguageModeling,
    )
    from datasets import Dataset
    print("✅ All packages ready\n")
except ImportError as e:
    print(f"❌ Missing: {e}")
    print("Install: pip install -q transformers datasets")
    sys.exit(1)

In [ ]:
# ============ CELL 3: CONFIGURATION ============
CONFIG = {
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,  # Small per-device batch
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 16,  # Effective batch = 1*16 = 16
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 1024,
    "bf16": True,  # Use bfloat16 (not quantized)
    "fp16": False,
    "output_dir": "medical_qwen_cpt_final",
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "logging_dir": "logs_cpt_final",
    "logging_steps": 5,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "seed": 42,
}

print("Configuration:")
print("="*70)
for k, v in CONFIG.items():
    print(f"{k:.<50} {v}")
print("="*70)
print(f"\nEffective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"Expected GPU memory: 26-30GB\n")

In [ ]:
# ============ CELL 4: LOAD DATA ============
def load_jsonl(file_path, max_samples=None):
    data = []
    if not Path(file_path).exists():
        raise FileNotFoundError(f"{file_path} not found")
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(CONFIG["train_file"])
eval_data = load_jsonl(CONFIG["eval_file"])

train_tokens = sum(c.get('token_count', 0) for c in train_data)
eval_tokens = sum(c.get('token_count', 0) for c in eval_data)

print(f"✅ Train: {len(train_data):,} chunks ({train_tokens:,} tokens)")
print(f"✅ Eval:  {len(eval_data):,} chunks ({eval_tokens:,} tokens)")
print(f"\nSample: {train_data[0]['text'][:200]}...\n")

In [ ]:
# ============ CELL 5: LOAD TOKENIZER ============
print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded\n")

In [ ]:
# ============ CELL 6: LOAD MODEL (FULL PRECISION, NO QUANTIZATION) ============
print(f"Loading model {CONFIG['model_name']}...")
print("(This may take 2-3 minutes)\n")

# Load in bfloat16 (NOT quantized - we need full model for CPT)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.bfloat16,  # Use bfloat16, not 8-bit quantization
    device_map="auto",
    trust_remote_code=True,
)

# CRITICAL: Enable gradient checkpointing to save memory
# This trades compute for memory - recompute activations during backward pass
model.gradient_checkpointing_enable()

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded")
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Precision: bfloat16 (full model, not quantized)")
print(f"   Gradient checkpointing: Enabled")
print(f"   Dtype: {next(model.parameters()).dtype}\n")

torch.cuda.synchronize()
gpu_mem_used = torch.cuda.memory_allocated(0) / 1e9
print(f"GPU memory after model load: {gpu_mem_used:.1f}GB\n")

In [ ]:
# ============ CELL 7: TOKENIZE DATASETS ============
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing datasets...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ CELL 8: SETUP DATA COLLATOR ============
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)
print("✅ Data collator configured\n")

In [ ]:
# ============ CELL 9: SETUP TRAINING ARGUMENTS ============
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    bf16=CONFIG["bf16"],
    fp16=CONFIG["fp16"],
    save_strategy=CONFIG["save_strategy"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    metric_for_best_model="eval_loss",
    logging_dir=CONFIG["logging_dir"],
    logging_steps=CONFIG["logging_steps"],
    seed=CONFIG["seed"],
    dataloader_num_workers=CONFIG["dataloader_num_workers"],
    dataloader_pin_memory=CONFIG["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)
print("✅ Training arguments configured\n")

In [ ]:
# ============ CELL 10: CREATE TRAINER ============
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)
print("✅ Trainer created and ready")
print("\n" + "="*80)
print("🚀 READY TO TRAIN (FULL MODEL)")
print("="*80)
print(f"Model: Full Qwen2.5-7B (7.62B parameters)")
print(f"Training: Continued Pretraining (CPT) on medical data")
print(f"Expected duration: 10-14 hours")
print(f"Output: {CONFIG['output_dir']}/")
print("="*80 + "\n")

In [ ]:
# ============ CELL 11: START TRAINING ============
print(f"\n🚀 Starting training: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    train_result = trainer.train()
    print(f"\n✅ Training complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Training loss: {train_result.training_loss:.4f}\n")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============ CELL 12: EVALUATE ============
print("\nEvaluating...\n")
eval_results = trainer.evaluate()
print("Results:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print()

In [ ]:
# ============ CELL 13: SAVE MODEL ============
best_model_path = Path(CONFIG["output_dir"]) / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

print(f"Saving model...")
trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

model_size = sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9
print(f"✅ Model saved to {best_model_path}")
print(f"   Size: {model_size:.2f}GB\n")

In [ ]:
# ============ CELL 14: TEST INFERENCE ============
print("Testing inference...\n")
from transformers import pipeline

inference_model = AutoModelForCausalLM.from_pretrained(
    str(best_model_path),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
)

prompts = [
    "The pathophysiology of venous insufficiency",
    "Duplex ultrasound is important for",
    "Treatment of varicose veins includes",
]

for prompt in prompts:
    output = generator(prompt, max_length=100, do_sample=True, temperature=0.7)
    print(f"Q: {prompt}")
    print(f"A: {output[0]['generated_text']}\n")

print("✅ Inference test complete\n")

In [ ]:
# ============ CELL 15: SUMMARY ============
print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)
print(f"\nBest model: {best_model_path}")
print(f"Size: {model_size:.2f}GB")
print(f"\nEvaluation:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
print(f"\nUsage:")
print(f"  from transformers import AutoModelForCausalLM")
print(f"  model = AutoModelForCausalLM.from_pretrained('{best_model_path}')")
print(f"\n" + "="*80)